# ETL del archivo en crudo `crew.parquet`

## Librerías

In [44]:
import os
import ast
import gc

import pandas as pd

## Extracción

In [45]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/crew.parquet?raw=true"

crew = (
    pd.read_parquet(
        url, 
        engine='fastparquet'
        )
    )

## Exploración

Se explora el dataframe.

In [46]:
crew

,crew,id
0,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...
45471,"[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45472,"[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45473,"[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45474,"[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


Ver un dato de la columna 'crew'. Es una cadena con la forma de una lista de diccionarios.

In [47]:
crew.iloc[0]['crew']

'[{\'credit_id\': \'52fe4284c3a36847f8024f49\', \'department\': \'Directing\', \'gender\': 2, \'id\': 7879, \'job\': \'Director\', \'name\': \'John Lasseter\', \'profile_path\': \'/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f4f\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12891, \'job\': \'Screenplay\', \'name\': \'Joss Whedon\', \'profile_path\': \'/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f55\', \'department\': \'Writing\', \'gender\': 2, \'id\': 7, \'job\': \'Screenplay\', \'name\': \'Andrew Stanton\', \'profile_path\': \'/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f5b\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12892, \'job\': \'Screenplay\', \'name\': \'Joel Cohen\', \'profile_path\': \'/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f61\', \'department\': \'Writing\', \'gender\': 0, \'id\': 12893, \'job\': \'Screenplay\', \'name\': \'A

Se obtiene una información general.

In [48]:
crew.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   crew    45476 non-null  object
 1   id      45476 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 710.7+ KB


Los datos están completos.

In [49]:
crew.isnull().sum()

crew    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay duplicados.

In [50]:
crew['id'].duplicated(
    keep='first'
    ).sum()

44

Se eliminan los duplicados.

In [51]:
crew.drop_duplicates(
    subset='id', 
    inplace=True
    )

Hay valores únicos.

In [52]:
crew['id'].duplicated(
    keep='first'
    ).sum()

0

### Renombrar el nombre de la columna 'id' por 'movie_id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con el dataset 'movies.parquet'.

In [53]:
crew.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

In [54]:
crew.columns

Index(['crew', 'movie_id'], dtype='object')

### Desanidar la columna 'crew'

Se convierte a las cadenas en listas de diccionarios.

In [55]:
crew['crew'] = (
    crew['crew']
    .apply(ast.literal_eval)
    )

Se separan los elementos de las listas en filas.

In [56]:
crew_en_filas = (
    crew.explode('crew')
    )

Se convierten las llaves en columnas.

In [57]:
crew_en_columnas = (
    pd.json_normalize(
        crew_en_filas['crew']
        )
    )

Se crea un dataframe con la columna 'movie_id' y las nuevas columnas.

In [58]:
crew = (
    crew_en_filas.drop(
        columns='crew'
        )
    .join(
        crew_en_columnas
        )
    )

Se eliminan los siguientes objetos para liberar memoria.

In [59]:
del crew_en_filas
del crew_en_columnas
gc.collect()

187

## Exploración de los datos desanidados

Se explora el dataframe.

In [60]:
crew

,movie_id,credit_id,department,gender,id,job,name,profile_path
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2.0,7879.0,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
...,...,...,...,...,...,...,...,...
45473,67758,52fe424ec3a36847f801409f,Art,2.0,1049.0,Production Design,Charles Rosen,None
45473,67758,52fe424ec3a36847f801409f,Art,2.0,1049.0,Production Design,Charles Rosen,None
45474,227506,52fe424ec3a36847f80140a5,Art,2.0,8867.0,Set Decoration,Marvin March,None
45474,227506,52fe424ec3a36847f80140a5,Art,2.0,8867.0,Set Decoration,Marvin March,None


Primera fila.

In [61]:
crew.iloc[0]

movie_id                                     862
credit_id               52fe4284c3a36847f8024f49
department                             Directing
gender                                       2.0
id                                        7879.0
job                                     Director
name                               John Lasseter
profile_path    /7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
Name: 0, dtype: object

Se obtiene una informacion general.

In [62]:
crew.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 464607 entries, 0 to 45475
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   movie_id      464607 non-null  int64  
 1   credit_id     464250 non-null  object 
 2   department    464250 non-null  object 
 3   gender        464250 non-null  float64
 4   id            464250 non-null  float64
 5   job           464250 non-null  object 
 6   name          464250 non-null  object 
 7   profile_path  107956 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 31.9+ MB


### Eliminar las columnas innecesarias

Se hace esto porque son innecesarias en el contexto de la visualizacion de las peliculas por parte de los usuarios.

In [63]:
innecesarias = [
    'credit_id', 
    'profile_path'
]

crew.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [64]:
set(crew.columns).isdisjoint(
     set(innecesarias)
     )

True

In [65]:
for columna in crew.columns:
    print(columna)

movie_id
department
gender
id
job
name


### Eliminar nulos en la columna 'name'

In [66]:
crew['name'].isnull().sum()

357

In [67]:
crew.dropna(
    subset='name', 
    inplace=True
    )

In [68]:
crew['name'].isnull().sum()

0

### Renombrar la columna 'id'

Se hace esto para que se identifique rapidamente que es el id de la persona.

In [69]:
crew.rename(
    columns={'id': 'person_id'}, 
    inplace=True
    )

In [70]:
('id' not in crew.columns 
 and 'person_id' in crew.columns)

True

### Cambiar los tipos de algunas columnas

Se hace esto porque los id son etiquetas y gender es un dato categorico. El resto de los datos tienen el tipo correcto que es str.

In [71]:
crew.dtypes

movie_id        int64
department     object
gender        float64
person_id     float64
job            object
name           object
dtype: object

Columna 'gender'.

In [72]:
crew['gender'].dtype

dtype('float64')

In [73]:
crew['gender'] = (
    crew['gender']
    .astype(str)
    )

In [74]:
crew['gender'].dtype

dtype('O')

Columna 'movie_id'

In [75]:
crew['movie_id'].dtype

dtype('int64')

In [76]:
crew['movie_id'] = (
    crew['movie_id']
    .astype(str)
    )

In [77]:
crew['movie_id'].dtype

dtype('O')

Columna 'person_id'

In [78]:
crew['person_id'].dtype

dtype('float64')

In [79]:
crew['person_id'] = (
    crew['person_id']
    .astype(str)
    )

In [80]:
crew['person_id'].dtype

dtype('O')

### Resetear los índices

In [81]:
crew.index

Int64Index([    0,     0,     0,     0,     0,     0,     0,     0,     0,
                0,
            ...
            45472, 45472, 45473, 45473, 45473, 45473, 45473, 45474, 45474,
            45475],
           dtype='int64', length=464250)

In [82]:
crew.reset_index(
    drop=True, 
    inplace=True
    )

In [83]:
crew.index

RangeIndex(start=0, stop=464250, step=1)

### Última revisión

In [84]:
crew.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 464250 entries, 0 to 464249
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   movie_id    464250 non-null  object
 1   department  464250 non-null  object
 2   gender      464250 non-null  object
 3   person_id   464250 non-null  object
 4   job         464250 non-null  object
 5   name        464250 non-null  object
dtypes: object(6)
memory usage: 21.3+ MB


Los datos están completos.

In [85]:
crew.isnull().sum()

movie_id      0
department    0
gender        0
person_id     0
job           0
name          0
dtype: int64

## Carga

In [86]:
ruta_actual = (
    os.getcwd()
    )
ruta_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [87]:
ruta_del_proyecto = (
    os.path.dirname(
        os.path.dirname(
            ruta_actual
            )
        )
    )
ruta_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [88]:
ruta_a_exportar = (
    os.path.join(
        ruta_del_proyecto, 
        'data', 
        'ETL', 
        'crew.parquet'
        )
    )
ruta_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\crew.parquet'

In [89]:
crew.to_parquet(
    ruta_a_exportar
    )

Se elimina el dataframe para liberar memoria.

In [90]:
del crew
gc.collect()

16